# Ejercicio 4 — Diseño $3^3$ completo y análisis RSM (3 factores)

**Objetivo.** Analizar un diseño $3^3$ completo (27 corridas) con ANOVA por componentes
lineal/cuadrático, ajustar el modelo de segundo orden multivariable y localizar el punto
óptimo mediante análisis canónico.

**Factores:**
- $A$ = Relación agua/cemento: 0.40 (−1), 0.49 (0), 0.57 (+1)  [real: 140, 170, 200 kg/m³ agua]
- $B$ = Contenido de cemento: 300 (−1), 350 (0), 400 kg/m³ (+1)
- $C$ = Contenido de arena: 600 (−1), 700 (0), 800 kg/m³ (+1)

**Respuesta:** Resistencia a compresión (MPa) a 28 días

**Dataset:** `../../datos/resistencia-concreto-3k3.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import linalg

df = pd.read_csv('../../datos/resistencia-concreto-3k3.csv')
print(f'Corridas: {len(df)} (3^3 = 27 ✓)' if len(df)==27 else f'Corridas: {len(df)}')
print(df.head())

## 1. ANOVA con tabla de grados de libertad $3^3$

In [ ]:
# Modelo completo de 2° orden
formula = 'resistencia ~ x1+x2+x3+I(x1**2)+I(x2**2)+I(x3**2)+x1:x2+x1:x3+x2:x3'
modelo = smf.ols(formula, data=df).fit()

anova_t1 = sm.stats.anova_lm(modelo, typ=1)
print('ANOVA tipo I — Modelo de 2° orden:')
print(anova_t1.round(4))
print(f'\nTotal GL: {anova_t1["df"].sum():.0f}  (esperado: 26)')

## 2. Descomposición L/Q por factor

Calculamos las medias marginales por nivel de cada factor y sus contrastes lineal (L)
y cuadrático (Q).

In [ ]:
def contrastes_lq(df, factor, respuesta='resistencia'):
    medias = df.groupby(factor)[respuesta].mean()
    y_neg1, y_0, y_pos1 = medias[-1], medias[0], medias[1]
    C_L = y_pos1 - y_neg1
    C_Q = y_neg1 - 2*y_0 + y_pos1
    n   = df.groupby(factor)[respuesta].count()[-1]  # observaciones por nivel
    # Con C_L calculado de medias: SC = n_nivel * C_medias² / Σc²  (Σc²=2 para L, 6 para Q)
    SC_L = n * C_L**2 / 2
    SC_Q = n * C_Q**2 / 6
    return {'Factor': factor, 'C_L': round(C_L,3), 'C_Q': round(C_Q,3),
            'SC_L': round(SC_L,3), 'SC_Q': round(SC_Q,3)}

res = [contrastes_lq(df, f) for f in ['x1','x2','x3']]
tabla_lq = pd.DataFrame(res)
tabla_lq.index = ['A (agua)','B (cemento)','C (arena)']
print('Tabla de contrastes L/Q:')
print(tabla_lq)
print('\nVerificación: compare SC_L + SC_Q de cada factor con su SS en el ANOVA tipo I.')

## 3. Resumen del modelo y coeficientes

In [ ]:
print(modelo.summary())

## 4. Análisis canónico: punto estacionario

El punto estacionario satisface $\nabla\hat{y} = 0$, es decir:
$(B \mathbf{x}_s + \mathbf{b}/2 = \mathbf{0})$, donde $B$ es la matriz de coeficientes
cuadráticos.

In [ ]:
p = modelo.params
b_vec = np.array([p['x1'], p['x2'], p['x3']]) / 2
B_mat = np.array([
    [p['I(x1 ** 2)'],     p['x1:x2']/2,  p['x1:x3']/2],
    [p['x1:x2']/2,    p['I(x2 ** 2)'],   p['x2:x3']/2],
    [p['x1:x3']/2,    p['x2:x3']/2,   p['I(x3 ** 2)']]
])

x_s = -linalg.solve(B_mat, b_vec)
y_s = (p['Intercept'] + p['x1']*x_s[0] + p['x2']*x_s[1] + p['x3']*x_s[2]
       + p['I(x1 ** 2)']*x_s[0]**2 + p['I(x2 ** 2)']*x_s[1]**2
       + p['I(x3 ** 2)']*x_s[2]**2
       + p['x1:x2']*x_s[0]*x_s[1] + p['x1:x3']*x_s[0]*x_s[2]
       + p['x2:x3']*x_s[1]*x_s[2])

# Unidades reales
centros_r = np.array([170, 350, 700])
deltas_r  = np.array([ 30,  50, 100])
x_real = centros_r + x_s * deltas_r

eigenvalores = np.linalg.eigvalsh(B_mat)

print('Punto estacionario (codificado):', x_s.round(3))
print('Punto estacionario (real): agua={:.0f}, cemento={:.0f}, arena={:.0f}'.format(*x_real))
print(f'Resistencia estimada: {y_s:.2f} MPa')
print('Eigenvalores de B:', eigenvalores.round(4))
tipo = 'MÁXIMO' if all(eigenvalores < 0) else ('MÍNIMO' if all(eigenvalores > 0) else 'SILLA')
print(f'Tipo de punto estacionario: {tipo}')

# ── Verificación de extrapolación ────────────────────────────────────────────
nombres_f = ['agua  (x1)', 'cemento (x2)', 'arena (x3)']
rango_real = [(140, 200), (300, 400), (600, 800)]
fuera = np.abs(x_s) > 1
if any(fuera):
    print('\n⚠  ADVERTENCIA — Extrapolación detectada:')
    for f, xv, rv, (lo, hi), out in zip(nombres_f, x_s, x_real, rango_real, fuera):
        marca = f'  ← FUERA del rango [{lo}–{hi}]' if out else ''
        print(f'   {f}: x = {xv:+.3f},  real = {rv:.0f}{marca}')
    print('\n   El modelo es una interpolación válida solo para |x| ≤ 1.')
    print('   Recomienda centrar un nuevo diseño en esta región y')
    print('   confirmar experimentalmente antes de adoptar estas condiciones.')
else:
    print('\n✓  El punto estacionario está dentro de la región experimental (|x| ≤ 1).')

## 5. Gráficos de curvas de nivel (fijando el tercer factor en su óptimo)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pares = [('x1','x2','x3'), ('x1','x3','x2'), ('x2','x3','x1')]
etiquetas = [('x₁ (Agua)', 'x₂ (Cemento)'), ('x₁ (Agua)', 'x₃ (Arena)'), ('x₂ (Cemento)', 'x₃ (Arena)')]

for ax, (f1, f2, f3), (e1, e2) in zip(axes, pares, etiquetas):
    g1, g2 = np.meshgrid(np.linspace(-1.5, 1.5, 50), np.linspace(-1.5, 1.5, 50))
    # Fijar f3 en su valor óptimo
    f3_val = x_s[['x1','x2','x3'].index(f3)]
    g = pd.DataFrame({f1: g1.ravel(), f2: g2.ravel(), f3: f3_val})
    zg = modelo.predict(g).values.reshape(g1.shape)
    cp = ax.contourf(g1, g2, zg, levels=12, cmap='RdYlGn')
    plt.colorbar(cp, ax=ax, label='Resistencia (MPa)')
    ax.set_xlabel(e1, fontsize=10)
    ax.set_ylabel(e2, fontsize=10)
    ax.set_title(f'{f3} fijado en {f3_val:.2f}', fontsize=10)
    ax.plot(x_s[['x1','x2','x3'].index(f1)],
            x_s[['x1','x2','x3'].index(f2)], 'r*', markersize=15)

plt.suptitle('Diseño $3^3$ — Curvas de nivel (par de factores, tercer factor en óptimo)',
             fontsize=12)
plt.tight_layout()
plt.show()

## 6. Conclusión

- El $3^3$ con 27 corridas permite estimar todos los efectos principales (L y Q) y las
  interacciones de dos factores (6 términos bilineales) sin réplica.
- El análisis canónico revela si el punto estacionario es un máximo, mínimo o punto de silla;
  en este caso los tres eigenvalores de $B$ son negativos, confirmando un **máximo**.
- Los gráficos de curvas de nivel (fijando un factor en su óptimo) facilitan la interpretación
  de la superficie cuando hay 3 factores.
- **Extrapolación:** el óptimo calculado sitúa el cemento en 434 kg/m³ ($x_2 = 1.67$),
  fuera del rango experimental (300–400 kg/m³, $|x| \leq 1$). En esa zona el modelo
  **extrapola** y sus predicciones no están validadas. El paso correcto es centrar un nuevo
  diseño (CCD o $3^2$ sobre $x_1$-$x_2$) alrededor de ese punto y confirmar el óptimo
  experimentalmente antes de implementar las condiciones.
- Con 27 corridas el $3^3$ es costoso; un CCD en 3 factores requiere solo $2^3 + 2\cdot3 + n_c$
  corridas (típicamente 15–17), manteniendo similar capacidad de ajuste.